# Práctica de la sesión 3 · Antes de unir, cuenta
**Estud-IA · Análisis de Datos Básico** · datos inventados de la Papelería Punto y Coma

Vamos celda por celda, juntos. Antes de correr cada consulta, **escribe en el chat cuántas filas crees que van a salir**.

Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

In [ ]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

ventas = cargar("https://raw.githubusercontent.com/santiagom26cc/ejemplo-proyecto-datos-basico/main/datos/originales/ventas.csv")
productos = cargar("https://raw.githubusercontent.com/santiagom26cc/ejemplo-proyecto-datos-basico/main/datos/originales/productos.csv")

# Dos copias del catálogo para practicar (no se guardan en ningún lado)
productos_dup = pd.concat([productos, pd.DataFrame([{"producto_id": "P1", "nombre": "Cuaderno grande",
                                                      "categoria": "Papelería", "costo_unitario": 6500}])],
                          ignore_index=True)
productos_sin_p4 = productos[productos["producto_id"] != "P4"].copy()
print("Listo: tablas ventas, productos, productos_dup y productos_sin_p4.")

## Paso 1 · Cuenta cada tabla
¿Cuántas filas tiene cada una? (8 y 4)

In [ ]:
sql("SELECT COUNT(*) AS filas_ventas FROM ventas")

In [ ]:
sql("SELECT COUNT(*) AS filas_productos FROM productos")

## Paso 2 · ¿La clave del catálogo es única?
Si `producto_id` se repite, la unión va a multiplicar filas. Esta consulta muestra los códigos repetidos: si no sale nada, la clave es única.

In [ ]:
sql("""
SELECT producto_id, COUNT(*) AS veces
FROM productos
GROUP BY producto_id
HAVING COUNT(*) > 1
""")

## Paso 3 · Predice y une
Cada venta busca su producto en el catálogo. ¿Cuántas filas salen? (8: una por venta)

In [ ]:
sql("""
SELECT v.id_venta, v.producto_id, p.nombre, p.categoria, v.cantidad, v.precio_unitario, v.estado
FROM ventas AS v
LEFT JOIN productos AS p ON v.producto_id = p.producto_id
ORDER BY v.id_venta
""")

## Paso 4 · Total de las ventas completadas, después de unir
Debe dar lo mismo que sin unir: **121.000**.

In [ ]:
sql("""
SELECT COUNT(*) AS filas, SUM(v.cantidad * v.precio_unitario) AS pesos
FROM ventas AS v
LEFT JOIN productos AS p ON v.producto_id = p.producto_id
WHERE v.estado = 'completada'
""")

## Paso 5 · ¿Y si el catálogo repite P1?
En `productos_dup`, P1 aparece dos veces (alguien reutilizó el código para un cuaderno grande). **Predice en el chat** cuántas filas salen y cuánto suman las completadas.

In [ ]:
sql("""
SELECT COUNT(*) AS filas,
       COUNT(*) FILTER (WHERE v.estado = 'completada') AS completadas,
       SUM(v.cantidad * v.precio_unitario) FILTER (WHERE v.estado = 'completada') AS pesos
FROM ventas AS v
LEFT JOIN productos_dup AS p ON v.producto_id = p.producto_id
""")

Salen **10 filas** y **171.000**: las ventas de P1 (V1 y V5) se cuentan dos veces y el total sube 50.000.

## Paso 6 · DISTINCT no lo arregla
Las dos filas de P1 no son copias exactas (cambia el nombre), así que DISTINCT las deja. La solución es revisar la clave, no esconder el problema.

In [ ]:
sql("""
SELECT COUNT(*) AS filas_con_distinct
FROM (
    SELECT DISTINCT *
    FROM ventas AS v
    LEFT JOIN productos_dup AS p ON v.producto_id = p.producto_id
)
""")

## Paso 7 · LEFT o INNER cuando falta un producto
En `productos_sin_p4` no está la bolsa (P4). **LEFT** conserva todas las ventas; **INNER** solo las que encuentran producto. ¿Qué población quieres?

In [ ]:
sql("""
SELECT 'LEFT' AS union_usada, COUNT(*) AS filas,
       SUM(v.cantidad * v.precio_unitario) FILTER (WHERE v.estado = 'completada') AS pesos
FROM ventas AS v LEFT JOIN productos_sin_p4 AS p ON v.producto_id = p.producto_id
UNION ALL
SELECT 'INNER', COUNT(*),
       SUM(v.cantidad * v.precio_unitario) FILTER (WHERE v.estado = 'completada')
FROM ventas AS v INNER JOIN productos_sin_p4 AS p ON v.producto_id = p.producto_id
ORDER BY union_usada DESC
""")

LEFT: 8 filas y 121.000. INNER: 6 filas y **113.000**: se perdió una venta completada de 8.000 (V8). Con LEFT, esa venta queda **sin categoría** (vacío), que no es lo mismo que cero:

In [ ]:
sql("""
SELECT p.categoria, SUM(v.cantidad * v.precio_unitario) AS pesos
FROM ventas AS v LEFT JOIN productos_sin_p4 AS p ON v.producto_id = p.producto_id
WHERE v.estado = 'completada'
GROUP BY p.categoria
ORDER BY pesos DESC
""")

## Paso 8 · Resumen por categoría y cifra de control
Con el catálogo bueno: Papelería 68.000 y Hogar 53.000. **La suma de los grupos debe dar el total (121.000).**

In [ ]:
sql("""
SELECT p.categoria, COUNT(*) AS ventas, SUM(v.cantidad) AS unidades,
       SUM(v.cantidad * v.precio_unitario) AS pesos
FROM ventas AS v
LEFT JOIN productos AS p ON v.producto_id = p.producto_id
WHERE v.estado = 'completada'
GROUP BY p.categoria
ORDER BY pesos DESC
""")

In [ ]:
sql("""
SELECT SUM(pesos) AS suma_de_los_grupos
FROM (
    SELECT p.categoria, SUM(v.cantidad * v.precio_unitario) AS pesos
    FROM ventas AS v
    LEFT JOIN productos AS p ON v.producto_id = p.producto_id
    WHERE v.estado = 'completada'
    GROUP BY p.categoria
)
""")

## Ampliación · WHERE filtra filas, HAVING filtra grupos
WHERE se aplica antes de agrupar; HAVING, después. Solo Papelería pasa de 60.000.

In [ ]:
sql("""
SELECT p.categoria, SUM(v.cantidad * v.precio_unitario) AS pesos
FROM ventas AS v
LEFT JOIN productos AS p ON v.producto_id = p.producto_id
WHERE v.estado = 'completada'
GROUP BY p.categoria
HAVING SUM(v.cantidad * v.precio_unitario) > 60000
""")